# Pocket Detection Method Comparison

Here we compare several pocket detection methods to choose an appropriate method for the validation study of T2FPharm pharmacophore modeling.

## TLDR

- All three DoGSite variants (DoGSiteScorer, and DoGSite3 with/without ligand bias) report higher (between 20 to 30%)   ligand coverage than the actual coverage calculated from the voxel maps.
- No variant could produce pockets with near 100% ligand coverage (highest was 75% by DoGSiteScorer).
- DoGSite3 produces pockets with internal holes and artifacts that are not attached to the pocket, further contributing to lower ligand coverage.

**Variables**

In [1]:
data_dirpath = "method_comparison_data"

**Notebook Settings**

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows

**Imports**

In [3]:
from pathlib import Path
import pickle
from typing import Literal

import numpy as np
import scipy as sp

import scids
import caddpy
import t2fpharm_study

**Helper Functions**

In [4]:
class PocketComparison:
    def __init__(
        self,
        entry: pd.Series,
    ):
        pdb_id = entry["pdb_id"]
        self.entry = entry
        self.complex = manager.complex(pdb_id)
        self.receptor = manager.receptor(pdb_id)
        self.ligand_atom_coords, self.ligand_mask = self._get_ligand_atom_coordinates()
        return

    def _get_ligand_atom_coordinates(self) -> tuple[np.ndarray, pd.Series]:
        """Get the coordinates of all ligand-of-interest atoms.

        This is used to calculate pocket coverage.
        """
        atoms = self.complex.composition.atoms
        selection = (
            (atoms["res_name"] == self.entry["ligand_res_name"])
            & (atoms["chain_id"] == self.entry["ligand_chain_id"])
            & (atoms["res_seq"] == self.entry["ligand_res_seq"])
        )
        ligand = self.complex.select(selection)
        return ligand.trajectory.points, selection


class LigandPocket(PocketComparison):
    def __init__(
        self,
        entry: pd.Series,
        radii_offset: float = 2.5,
        grid: float | scids.grid.Grid = 0.3
    ):
        super().__init__(entry=entry)
        self.best_pocket = caddpy.pocket.from_ligand(
            system=self.complex,
            ligand_mask=self.ligand_mask,
            ligand_radii_offset=radii_offset
        )
        ligand_atom_is_covered = self.best_pocket.point_coverage(self.ligand_atom_coords)
        self.coverage = 100 * np.count_nonzero(ligand_atom_is_covered) / ligand_atom_is_covered.size
        self.best_pocket_holes = self.best_pocket.holes()
        self.best_pocket_has_holes = np.any(self.best_pocket_holes)
        return
        

class DoGSitePocket(PocketComparison):
    def __init__(
        self,
        entry: pd.Series,
        algorithm: Literal["scorer", "3"],
        ligand_bias: bool = False,
    ):
        super().__init__(entry=entry)
        self.pockets = self._get_dogsite_pockets(algorithm=algorithm, ligand_bias=ligand_bias)
        self.coverage = self.pockets.point_coverage(self.ligand_atom_coords)
        self.coverage_comparison = self._compare_coverage()
        self.best_pocket = self._select_best_pocket()
        self.best_pocket_holes = self.best_pocket.holes()
        self.best_pocket_has_holes = np.any(self.best_pocket_holes)
        return
        
    def _get_dogsite_pockets(
        self,
        algorithm: Literal["scorer", "3"],
        ligand_bias: bool = False,
    ) -> caddpy.pocket.Pockets:
        """Get pockets from DoGSite web API."""
        filename = (
            f"{self.entry["pdb_id"]}-dogsite{algorithm}"
            f"{f"-{"biased" if ligand_bias else "nonbiased"}" if algorithm == "3" else ""}.pkl"
        )
        filepath_pickle = Path(data_dirpath) / filename
        if not filepath_pickle.is_file():
            pockets = caddpy.pocket.from_dogsite(
                self.complex,
                chain_id=self.entry["chain_id"],
                ligand_id=(
                    self.entry["ligand_res_name"], 
                    self.entry["ligand_chain_id"], 
                    self.entry["ligand_res_seq"]
                ),
                include_subpockets=True,
                calculate_druggability=True,
                algorithm=algorithm,
                ligand_bias=ligand_bias,
            )
            with open(filepath_pickle, "wb") as f:
                pickle.dump(pockets, f)
        else:
            with open(filepath_pickle, "rb") as f:
                pockets = pickle.load(f)
        return pockets

    def _compare_coverage(self) -> pd.DataFrame:
        """Calculate and merge ligand coverage with the main dataframe."""
        
        # Calculate coverage percentage per pocket label
        df_coverage_percent = (self.coverage.mean() * 100).round(2).reset_index()
        df_coverage_percent.columns = ['label', 'calc_lig_cov']
    
        # Drop any index name on df_data so 'label' only lives in the column
        df_data = self.pockets.external_data.copy()
        df_data.index.name = None
        
        # Select only the needed columns from the main dataframe
        cols_to_keep = ['name', 'lig_cov', 'poc_cov', 'volume', 'label', 'parent_label']
        df_data = df_data[cols_to_keep]
    
        # Merge on 'label'
        df_merged = df_data.merge(df_coverage_percent, on='label', how='inner')
    
        # Sort by lig_cov descending
        df_sorted = df_merged.sort_values(by='lig_cov', ascending=False)
    
        # Reorder columns as requested
        final_order = [
            'name',
            'lig_cov',
            'calc_lig_cov',
            'poc_cov',
            'volume',
            'label',
            'parent_label'
        ]
        return df_sorted[final_order].reset_index(drop=True)

    def _select_best_pocket(self) -> caddpy.pocket.Pocket:
        """Select the pocket whose sum of ligand and pocket coverage is highest."""
        best_pocket_idx = (self.pockets.external_data["lig_cov"] + self.pockets.external_data["poc_cov"]).idxmax()
        best_pocket_label = self.pockets.external_data.loc[best_pocket_idx, "label"]
        return self.pockets.pockets.loc[best_pocket_label, "pocket"]


**Study Manager**

In [5]:
manager = t2fpharm_study.manager()

**Structure Selection**

We select one structure from the database as an example to perform different pocket detection methods on.

In [6]:
PDB_ID = "1AQ1"

In [7]:
entry = manager.dataset.loc[PDB_ID]

## DoGSiteScorer

In [8]:
dog = DoGSitePocket(entry=entry, algorithm="scorer")

General pocket data:

In [9]:
dog.pockets.pockets

,label,volume,point_count,is_subpocket,parent_label,pocket
label,,,,,,
1,1,2081.280296,32520,False,1,"Field(\n grid=Grid(\n shape=[74 74 53],\n ..."
2,2,361.792052,5653,False,2,"Field(\n grid=Grid(\n shape=[36 38 32],\n ..."
3,3,236.928034,3702,False,3,"Field(\n grid=Grid(\n shape=[28 24 27],\n ..."
4,4,208.832030,3263,False,4,"Field(\n grid=Grid(\n shape=[16 26 29],\n ..."
5,5,194.112028,3033,False,5,"Field(\n grid=Grid(\n shape=[18 28 23],\n ..."
6,6,187.264027,2926,False,6,"Field(\n grid=Grid(\n shape=[24 27 26],\n ..."
7,7,149.440021,2335,False,7,"Field(\n grid=Grid(\n shape=[16 24 28],\n ..."
8,8,126.272018,1973,False,8,"Field(\n grid=Grid(\n shape=[13 18 26],\n ..."
9,9,108.480015,1695,False,9,"Field(\n grid=Grid(\n shape=[14 20 17],\n ..."


Extra pocket data calculated by DoGSite:

In [10]:
dog.pockets.external_data

,name,lig_cov,poc_cov,lig_name,volume,enclosure,surface,depth,surf/vol,lid/hull,ellVol,ell c/a,ell b/a,siteAtms,accept,donor,hydrophobic_interactions,hydrophobicity,metal,Cs,Ns,Os,Ss,Xs,negAA,posAA,polarAA,apolarAA,ALA,ARG,ASN,ASP,CYS,GLN,GLU,GLY,HIS,ILE,LEU,LYS,MET,PHE,PRO,SER,THR,TRP,TYR,VAL,simpleScore,drugScore,center_x,center_y,center_z,max_radius,atom_serials,mrc,label,parent_label
label,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,P_0,92.06,18.26,STU_A_299,2081.28,0.12,2929.3,26.21,1.407451,-,-,0.27,0.84,381,111,53,142,0.46,0,275,49,57,0,0,0.13,0.16,0.23,0.48,5,3,2,4,0,2,6,5,4,4,11,6,0,4,4,0,6,1,3,9,0.67,0.814011,-6.83,27.23,14.46,19.64,"(163, 165, 166, 167, 169, 172, 176, 182, 185, ...",MrcFile(\n n_xyz = [78 78 57]\n mode ...,1,1
2,P_1,0.0,0.0,STU_A_299,361.79,0.14,675.61,12.55,1.867409,-,-,0.06,0.1,99,30,13,23,0.35,0,66,17,15,1,0,0.14,0.14,0.18,0.55,2,2,0,2,0,0,1,1,0,2,2,1,2,1,2,1,2,0,0,1,0.18,0.605527,17.28,24.05,21.45,13.09,"(1444, 1448, 1451, 1454, 1457, 1487, 1503, 150...",MrcFile(\n n_xyz = [40 42 36]\n mode ...,2,2
3,P_2,0.0,0.0,STU_A_299,236.93,0.2,541.23,13.42,2.284346,-,-,0.14,0.17,67,18,10,39,0.58,0,48,10,9,0,0,0.0,0.27,0.27,0.47,1,1,0,0,0,1,0,1,2,1,2,1,0,1,2,1,1,0,0,0,0.15,0.539674,18.07,25.59,10.76,8.63,"(1386, 1390, 1403, 1404, 1405, 1463, 1465, 146...",MrcFile(\n n_xyz = [32 28 31]\n mode ...,3,3
4,P_3,0.0,0.0,STU_A_299,208.83,0.22,382.14,9.52,1.829909,-,-,0.12,0.21,58,18,8,20,0.43,0,38,10,10,0,0,0.0,0.2,0.33,0.47,3,1,1,0,0,0,0,0,1,2,1,1,0,0,1,3,1,0,0,0,0.06,0.361334,-0.4,40.38,31.22,9.05,"(1883, 1884, 1898, 1900, 1902, 1903, 1904, 193...",MrcFile(\n n_xyz = [20 30 33]\n mode ...,4,4
5,P_4,0.0,0.0,STU_A_299,194.11,0.0,324.82,11.74,1.673381,-,-,0.23,0.35,82,22,12,8,0.19,0,57,14,11,0,0,0.12,0.25,0.19,0.44,1,2,0,2,0,0,0,0,1,1,2,1,0,0,0,1,1,1,1,2,0.0,0.435325,-1.97,25.51,25.5,8.11,"(2045, 2047, 2049, 2052, 2055, 2060, 2062, 206...",MrcFile(\n n_xyz = [22 32 27]\n mode ...,5,5
6,P_5,0.0,0.0,STU_A_299,187.26,0.26,342.24,13.75,1.827619,-,-,0.1,0.12,57,16,9,10,0.29,0,40,9,8,0,0,0.08,0.23,0.15,0.54,2,3,0,0,0,0,1,0,0,0,1,0,0,1,1,0,2,0,0,2,0.0,0.519625,11.86,18.57,31.64,9.78,"(3156, 3158, 3159, 3160, 3164, 3166, 3170, 321...",MrcFile(\n n_xyz = [28 31 30]\n mode ...,6,6
7,P_6,0.0,0.0,STU_A_299,149.44,0.14,302.2,13.91,2.022216,-,-,0.12,0.12,56,14,5,13,0.41,0,41,6,9,0,0,0.07,0.14,0.29,0.5,0,1,0,1,0,0,0,1,1,0,2,0,0,1,1,1,1,2,1,1,0.0,0.491851,6.63,26.84,42.87,8.61,"(3063, 3065, 3574, 3575, 3576, 3579, 3581, 358...",MrcFile(\n n_xyz = [20 28 32]\n mode ...,7,7
8,P_7,0.0,0.0,STU_A_299,126.27,0.27,238.09,8.92,1.885563,-,-,0.14,0.15,33,13,5,11,0.38,0,22,4,5,2,0,0.08,0.0,0.42,0.5,0,0,0,0,1,0,1,1,0,0,0,0,1,0,2,1,0,1,2,2,0.0,0.275457,-6.5,31.55,38.76,8.08,"(2813, 2816, 2819, 2821, 2892, 2895, 2923, 292...",MrcFile(\n n_xyz = [17 22 30]\n mode ...,8,8
9,P_8,0.0,0.0,STU_A_299,108.48,0.24,336.73,8.28,3.104074,-,-,0.33,0.52,30,10,0,22,0.69,0,23,2,5,0,0,0.29,0.14,0.0,0.57,0,0,0,0,0,0,2,0,1,0,3,0,0,0,0,0,0,0,0,1,0.02,0.24576,-10.04,42.81,18.33,5.73,"(808, 850, 852, 853, 854, 857, 859, 863, 867, ...",MrcFile(\n n_xyz = [18 24 21]\n mode ...,9,9


Calculated ligand coverage data:

In [11]:
dog.coverage

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
0,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


Comparison of ligand coverage data (DoGSite vs. here):

We see that the actual ligand coverage is about 20% lower and does not match the calculated value by DoGSite:

In [12]:
dog.coverage_comparison

,name,lig_cov,calc_lig_cov,poc_cov,volume,label,parent_label
0,P_0,92.06,75.41,18.26,2081.28,1,1
1,P_0_0,88.89,75.41,49.0,761.66,10,1
2,P_0_1,14.29,0.00,1.95,350.4,11,1
3,P_1,0.0,0.00,0.0,361.79,2,2
4,P_2,0.0,0.00,0.0,236.93,3,3
5,P_5,0.0,0.00,0.0,187.26,6,6
6,P_6,0.0,0.00,0.0,149.44,7,7
7,P_3,0.0,0.00,0.0,208.83,4,4
8,P_4,0.0,0.00,0.0,194.11,5,5
9,P_8,0.0,0.00,0.0,108.48,9,9


Does the pocket have holes?

In [13]:
dog.best_pocket_has_holes

Array(False, dtype=bool)

Display the pocket:

In [14]:
dog.best_pocket.display(wireframe=False).display(gui=True)

ThemeManager()

NGLWidget(gui_style='ngl')

## DoGSite3 (Non-Biased)

In [15]:
dog3 = DoGSitePocket(entry=entry, algorithm="3", ligand_bias=False)

General pocket data:

In [16]:
dog3.pockets.pockets

,label,volume,point_count,is_subpocket,parent_label,pocket
label,,,,,,
1,1,155.252776,298,False,1,"Field(\n grid=Grid(\n shape=[10 12 11],\n ..."
2,2,141.186249,271,False,2,"Field(\n grid=Grid(\n shape=[13 12 10],\n ..."
3,3,79.710318,153,False,3,"Field(\n grid=Grid(\n shape=[ 9 10 7],\n ..."
4,4,77.105405,148,False,4,"Field(\n grid=Grid(\n shape=[7 8 9],\n ..."
5,5,72.416563,139,False,5,"Field(\n grid=Grid(\n shape=[ 9 11 7],\n ..."
6,6,55.745124,107,False,6,"Field(\n grid=Grid(\n shape=[8 8 7],\n ..."
7,7,42.199580,81,False,7,"Field(\n grid=Grid(\n shape=[6 7 7],\n ..."
8,8,35.947790,69,False,8,"Field(\n grid=Grid(\n shape=[6 7 6],\n ..."
9,9,84.920142,163,True,2,"Field(\n grid=Grid(\n shape=[9 8 8],\n ..."


Extra pocket data calculated by DoGSite:

In [17]:
dog3.pockets.external_data

,name,lig_cov,poc_cov,lig_name,4A_crit,ligSASRatio,volume,enclosure,surface,lipoSurface,depth,surf/vol,lid/hull,ellVol,ell_c/a,ell_b/a,surfGPs,lidGPs,hullGPs,siteAtms,accept,donor,posAtms,negAtms,aromat,hydroAtms,hydrophobicity,metal,Cs,Ns,Os,Ss,Xs,acidicAA,basicAA,polarAA,apolarAA,sumAA,ALA,ARG,ASN,ASP,CYS,GLN,GLU,GLY,HIS,ILE,LEU,LYS,MET,PHE,PRO,SER,THR,TRP,TYR,VAL,A,C,G,U,I,N,DA,DC,DG,DT,DN,UNK,pdb,mrc,label,parent_label
label,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,P_1,91.4286,100,STU_A_299,1,0.131045,169.472,0.881679,285.993,180.493,9.08625,1.68755,0.118321,67.4507,0.230936,0.480827,231,31,262,31,5,1,0,1,5,19,0.806452,0,25,1,5,0,0,2,1,4,8,15,2,0,1,1,0,1,1,2,0,1,2,1,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,1,1
2,P_2,0.0,0,<NA>,0,0.0,154.624,0.957692,334.779,168.039,11.5655,2.16512,0.042308,78.1399,0.142987,0.224232,249,11,260,45,8,5,0,1,3,25,0.711111,0,32,5,8,0,0,1,2,2,12,17,3,2,0,1,0,0,0,0,0,2,4,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,2,2
3,P_3,0.0,0,<NA>,0,0.0,87.04,1.0,120.543,58.8672,7.41889,1.38491,0.0,20.5827,0.285281,0.498157,158,0,158,28,4,3,0,0,6,15,0.75,0,21,3,4,0,0,1,1,0,11,13,1,0,0,1,0,0,0,0,0,2,5,1,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,3,3
4,P_4,0.0,0,<NA>,0,0.0,85.504,0.873239,224.434,116.115,6.49923,2.62484,0.126761,15.728,0.489993,0.89611,124,18,142,27,8,7,0,2,1,10,0.518519,0,14,5,8,0,0,2,2,3,3,10,1,1,0,2,0,0,0,1,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,4,4
5,P_5,0.0,0,<NA>,0,0.0,78.336,0.96,202.528,102.985,8.61626,2.58538,0.04,16.7094,0.126523,0.19993,144,6,150,29,7,3,1,1,0,18,0.689655,0,20,2,7,0,0,2,3,2,5,12,1,2,0,1,0,0,1,1,0,1,0,1,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,5,5
6,P_6,0.0,0,<NA>,0,0.0,62.464,1.0,251.574,46.0519,6.14492,4.0275,0.0,8.00624,0.370862,0.472525,110,0,110,24,5,3,0,1,8,7,0.708333,0,17,2,5,0,0,1,1,3,3,8,1,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,6,6
7,P_7,0.0,0,<NA>,0,0.0,47.104,0.722892,195.469,103.357,4.59565,4.14974,0.277108,4.53331,0.538261,0.542249,60,23,83,16,4,4,1,0,0,7,0.5,0,8,4,4,0,0,0,1,2,7,10,0,1,0,0,0,0,0,1,0,2,1,0,1,0,2,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,7,7
8,P_8,48.5714,100,STU_A_299,1,0.131045,37.376,0.915493,184.935,86.9554,4.93153,4.94797,0.084507,3.01536,0.435312,0.64711,65,6,71,19,3,3,0,0,3,8,0.684211,0,13,3,3,0,0,1,1,1,4,7,0,0,0,1,0,1,0,0,1,1,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,8,8
9,P_2_1,0.0,0,<NA>,0,0.0,93.184,0.926174,295.924,141.014,6.8352,3.1757,0.073826,19.1117,0.55631,0.692279,138,11,149,33,7,4,0,1,2,16,0.666667,0,22,4,7,0,0,1,2,2,7,12,3,2,0,1,0,0,0,0,0,1,1,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,9,2


Calculated ligand coverage data:

In [18]:
dog3.coverage

,1,2,3,4,5,6,7,8,9,10
0,True,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False,False
8,True,False,False,False,False,False,False,False,False,False
9,False,False,False,False,False,False,False,False,False,False


Comparison of ligand coverage data (DoGSite vs. here):

We see that the actual ligand coverage is about 20% lower and does not match the calculated value by DoGSite:

In [19]:
dog3.coverage_comparison

,name,lig_cov,calc_lig_cov,poc_cov,volume,label,parent_label
0,P_1,91.4286,29.51,100,169.472,1,1
1,P_8,48.5714,6.56,100,37.376,8,8
2,P_2,0.0,0.00,0,154.624,2,2
3,P_3,0.0,0.00,0,87.04,3,3
4,P_5,0.0,0.00,0,78.336,5,5
5,P_4,0.0,0.00,0,85.504,4,4
6,P_6,0.0,0.00,0,62.464,6,6
7,P_7,0.0,0.00,0,47.104,7,7
8,P_2_1,0.0,0.00,0,93.184,9,2
9,P_2_2,0.0,0.00,0,61.44,10,2


Does the pocket have holes?

In [20]:
dog3.best_pocket_has_holes

Array(True, dtype=bool)

Display the pocket:

In [21]:
dog3.best_pocket.display(wireframe=False).display(gui=True)

NGLWidget(gui_style='ngl')

## DoGSite3 (Ligand Biased)

In [22]:
dog3b = DoGSitePocket(entry=entry, algorithm="3", ligand_bias=True)

General pocket data:

In [23]:
dog3b.pockets.pockets

,label,volume,point_count,is_subpocket,parent_label,pocket
label,,,,,,
1,1,765.323246,1469,False,1,"Field(\n grid=Grid(\n shape=[25 27 22],\n ..."
2,2,141.186249,271,False,2,"Field(\n grid=Grid(\n shape=[13 12 10],\n ..."
3,3,77.105405,148,False,3,"Field(\n grid=Grid(\n shape=[7 8 9],\n ..."
4,4,72.416563,139,False,4,"Field(\n grid=Grid(\n shape=[ 9 11 7],\n ..."
5,5,55.745124,107,False,5,"Field(\n grid=Grid(\n shape=[8 8 7],\n ..."
6,6,42.199580,81,False,6,"Field(\n grid=Grid(\n shape=[6 7 7],\n ..."
7,7,84.920142,163,True,2,"Field(\n grid=Grid(\n shape=[9 8 8],\n ..."
8,8,56.266107,108,True,2,"Field(\n grid=Grid(\n shape=[8 7 7],\n ..."


Extra pocket data calculated by DoGSite:

In [24]:
dog3b.pockets.external_data

,name,lig_cov,poc_cov,lig_name,4A_crit,ligSASRatio,volume,enclosure,surface,lipoSurface,depth,surf/vol,lid/hull,ellVol,ell_c/a,ell_b/a,surfGPs,lidGPs,hullGPs,siteAtms,accept,donor,posAtms,negAtms,aromat,hydroAtms,hydrophobicity,metal,Cs,Ns,Os,Ss,Xs,acidicAA,basicAA,polarAA,apolarAA,sumAA,ALA,ARG,ASN,ASP,CYS,GLN,GLU,GLY,HIS,ILE,LEU,LYS,MET,PHE,PRO,SER,THR,TRP,TYR,VAL,A,C,G,U,I,N,DA,DC,DG,DT,DN,UNK,pdb,mrc,label,parent_label
label,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,P_1,100,65.5401,STU_A_299,1,0.131045,848.384,0.940476,942.717,412.632,23.8529,1.11119,0.059524,4134.56,0.081041,0.446747,1185,75,1260,111,21,14,2,4,14,54,0.693694,0,77,13,21,0,0,6,7,7,20,40,2,0,1,2,0,2,4,2,1,3,10,6,0,3,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,1,1
2,P_2,0,0.0,<NA>,0,0.0,154.624,0.957692,334.779,168.039,11.5655,2.16512,0.042308,78.1399,0.142987,0.224232,249,11,260,45,8,5,0,1,3,25,0.711111,0,32,5,8,0,0,1,2,2,12,17,3,2,0,1,0,0,0,0,0,2,4,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,2,2
3,P_3,0,0.0,<NA>,0,0.0,85.504,0.873239,224.434,116.115,6.49923,2.62484,0.126761,15.728,0.489993,0.89611,124,18,142,27,8,7,0,2,1,10,0.518519,0,14,5,8,0,0,2,2,3,3,10,1,1,0,2,0,0,0,1,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,3,3
4,P_4,0,0.0,<NA>,0,0.0,78.336,0.96,202.528,102.985,8.61626,2.58538,0.04,16.7094,0.126523,0.19993,144,6,150,29,7,3,1,1,0,18,0.689655,0,20,2,7,0,0,2,3,2,5,12,1,2,0,1,0,0,1,1,0,1,0,1,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,4,4
5,P_5,0,0.0,<NA>,0,0.0,62.464,1.0,251.574,46.0519,6.14492,4.0275,0.0,8.00624,0.370862,0.472525,110,0,110,24,5,3,0,1,8,7,0.708333,0,17,2,5,0,0,1,1,3,3,8,1,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,5,5
6,P_6,0,0.0,<NA>,0,0.0,47.104,0.722892,195.469,103.357,4.59565,4.14974,0.277108,4.53331,0.538261,0.542249,60,23,83,16,4,4,1,0,0,7,0.5,0,8,4,4,0,0,0,1,2,7,10,0,1,0,0,0,0,0,1,0,2,1,0,1,0,2,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,6,6
7,P_2_1,0,0.0,<NA>,0,0.0,93.184,0.926174,295.924,141.014,6.8352,3.1757,0.073826,19.1117,0.55631,0.692279,138,11,149,33,7,4,0,1,2,16,0.666667,0,22,4,7,0,0,1,2,2,7,12,3,2,0,1,0,0,0,0,0,1,1,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,7,2
8,P_2_2,0,0.0,<NA>,0,0.0,61.44,1.0,109.275,52.1326,5.87878,1.77856,0.0,9.25248,0.511981,0.775717,111,0,111,18,2,2,0,0,3,11,0.777778,0,14,2,2,0,0,0,1,1,8,10,2,1,0,0,0,0,0,0,0,2,3,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,HEADER File contains complete residues of bind...,MrcFile(\n n_xyz = [74 57 87]\n mode ...,8,2


Calculated ligand coverage data:

In [25]:
dog3b.coverage

,1,2,3,4,5,6,7,8
0,True,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False
6,True,False,False,False,False,False,False,False
7,True,False,False,False,False,False,False,False
8,True,False,False,False,False,False,False,False
9,True,False,False,False,False,False,False,False


Comparison of ligand coverage data (DoGSite vs. here):

We see that the actual ligand coverage is about 20% lower and does not match the calculated value by DoGSite:

In [26]:
dog3b.coverage_comparison

,name,lig_cov,calc_lig_cov,poc_cov,volume,label,parent_label
0,P_1,100,67.21,65.5401,848.384,1,1
1,P_2,0,0.00,0.0,154.624,2,2
2,P_3,0,0.00,0.0,85.504,3,3
3,P_4,0,0.00,0.0,78.336,4,4
4,P_5,0,0.00,0.0,62.464,5,5
5,P_6,0,0.00,0.0,47.104,6,6
6,P_2_1,0,0.00,0.0,93.184,7,2
7,P_2_2,0,0.00,0.0,61.44,8,2


Does the pocket have holes?

In [27]:
dog3b.best_pocket_has_holes

Array(True, dtype=bool)

Display the pocket:

In [28]:
dog3b.best_pocket.display(wireframe=False).display(gui=True)

NGLWidget(gui_style='ngl')

## From Ligand

In [29]:
ligand_pocket = LigandPocket(entry=entry)

In [30]:
ligand_pocket.coverage

np.float64(100.0)

In [31]:
ligand_pocket.best_pocket_has_holes

Array(False, dtype=bool)

In [32]:
ligand_pocket.best_pocket.display(show_box=True).display(gui=True)

NGLWidget(gui_style='ngl')